In [1]:
import numpy as np
import tensorflow as tf

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.8
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from network import build_vggnet16
from train import WarmUpCosine, CustomWeightDecaySGD, LastNSaver, make_train_ds, make_test_ds, RSquared 

In [4]:
num_classes = 3
initial_lr = 0.1
weight_decay = 1e-4
epochs = 200
warmup_epochs = 5
batch_size = 128
image_size = 32

In [ ]:
SAVE_PATH = "utkface_12k_split.npz" 
def load_dataset_if_exists(path=SAVE_PATH):
    """
    Load if file exists, otherwise return None.
    """
    if os.path.exists(path):
        data = np.load(path)
        print("✔ Cached data detected, loading directly")
        return (data["x_train"], data["y_train"],
                data["x_test"], data["y_test"])
    return None

In [6]:
x_train, y_train, x_test, y_test = load_dataset_if_exists()
y_train = tf.cast(y_train, dtype=tf.float32)
y_test = tf.cast(y_test, dtype=tf.float32)

✔ 检测到缓存数据，已直接加载


In [7]:
NN_16=[32,32,64,64,128,128,256,256]

In [8]:
model = build_vggnet16(NN_16, num_class=1)

In [9]:
total_steps = epochs * (x_train.shape[0] // batch_size)
warmup_steps = warmup_epochs * (x_train.shape[0] // batch_size)
lr_schedule = WarmUpCosine(initial_lr, total_steps, warmup_steps)
optimizer = CustomWeightDecaySGD(
    weight_decay=weight_decay,
    learning_rate=lr_schedule,
    momentum=0.9,
    nesterov=False
)
model.compile(
    optimizer=optimizer,
    loss='mse',
    metrics=[RSquared()]
)

In [10]:
saver = LastNSaver(n=20)

In [11]:
train_ds = make_train_ds(
    x_train, y_train,
    batch_size=batch_size,
    image_size=32, pad=4,
    mixup_alpha=0.05   # 0.2~0.4 common; increase if overfitting
)
test_ds = make_test_ds(x_test, y_test, batch_size=batch_size)

model.fit(
    train_ds,
    epochs=epochs,
    validation_data=test_ds,
    verbose=2,
    callbacks=[saver]
)

Epoch 1/200
156/156 - 7s - loss: 0.0388 - r_squared: -3.0772e-01 - val_loss: 0.0479 - val_r_squared: -4.6725e-01 - 7s/epoch - 46ms/step
Epoch 2/200
156/156 - 2s - loss: 0.0232 - r_squared: 0.2284 - val_loss: 0.0280 - val_r_squared: 0.1417 - 2s/epoch - 16ms/step
Epoch 3/200
156/156 - 2s - loss: 0.0182 - r_squared: 0.3913 - val_loss: 0.0527 - val_r_squared: -6.1168e-01 - 2s/epoch - 16ms/step
Epoch 4/200
156/156 - 2s - loss: 0.0162 - r_squared: 0.4599 - val_loss: 0.0169 - val_r_squared: 0.4829 - 2s/epoch - 16ms/step
Epoch 5/200
156/156 - 2s - loss: 0.0147 - r_squared: 0.5030 - val_loss: 0.0523 - val_r_squared: -5.9902e-01 - 2s/epoch - 16ms/step
Epoch 6/200
156/156 - 2s - loss: 0.0128 - r_squared: 0.5681 - val_loss: 0.0467 - val_r_squared: -4.2957e-01 - 2s/epoch - 16ms/step
Epoch 7/200
156/156 - 2s - loss: 0.0120 - r_squared: 0.6034 - val_loss: 0.0151 - val_r_squared: 0.5384 - 2s/epoch - 16ms/step
Epoch 8/200
156/156 - 3s - loss: 0.0111 - r_squared: 0.6267 - val_loss: 0.0229 - val_r_square

In [12]:
model.save("VGG_11.h5")